In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("./src")

import numpy as np
import pandas as pd
from datetime import datetime as dt
import matplotlib.pyplot as plt
import nbformat

from data_loader import (
    load_secrets, get_tiingo_client, load_tbill_data, load_equity_market_data,
    get_option_inputs,
)
from pricing import black_scholes, delta, gamma, theta, vega, rho, build_greeks_table, build_bs_price_table
from monte_carlo import simulate_terminal, build_mc_price_table, mc_price
from implied_vol_BS import iv_NR, iv_brentq, iv_solve, build_iv_table
from option_api import fetch_option_chain

In [ ]:
DATA_DIR = "./data"

secrets = load_secrets(secrets_path="./secrets.json")
client = get_tiingo_client(secrets)

df_tbill = load_tbill_data(data_dir=DATA_DIR)
df_tbill.tail()



In [ ]:
symbol = "AMD"
df_symbol = load_equity_market_data(symbol, client)

optexp = dt(2026, 7, 31)   # expiration date matches the option chain queried below

inputs = get_option_inputs(df_symbol, df_tbill, optexp)
S = inputs["S"]
T = inputs["T"]
r = inputs["r"]
sigma = inputs["sigma"]
valuation_date = inputs["valuation_date"]
log_returns = inputs["log_returns"]

print(f"Spot price (S): {S:.2f}")
print(f"Time to expiration (T, years): {T:.4f}")
print(f"Risk-free rate (r): {r:.5f}")
print(f"Historical volatility (sigma): {sigma:.4f}")
print("Valuation date:", valuation_date)

In [ ]:
opt_chain, resolved_date = fetch_option_chain(
    symbol,
    optexp.strftime("%Y-%m-%d"),
    valuation_date,
    S,
    option_type="call",
    strike_pct_low=0.8,
    strike_pct_high=1.45,
)

print("Resolved valuation date:", resolved_date)
opt_chain[["strike", "bid", "ask", "Pmkt"]]

In [ ]:
K = opt_chain["strike"].tolist()
print("Strikes in range:", K)

In [ ]:
bs_prices = build_bs_price_table(S, K, T, r, sigma)
parity_check = bs_prices.copy()
parity_check["LHS (C - P)"] = parity_check["B-S Call"] - parity_check["B-S Put"]
parity_check["RHS (S - K*e^-rT)"] = S - np.array(K) * np.exp(-r * T)
parity_check["Diff"] = (parity_check["LHS (C - P)"] - parity_check["RHS (S - K*e^-rT)"]).abs()
parity_check

In [ ]:
mc_prices = build_mc_price_table(S, r, sigma, T, K, n_sims=10_000_000, seed=42)
mc_prices

In [ ]:
sim_sizes = [1_000, 10_000, 100_000, 1_000_000, 10_000_000]
strike_check = K[len(K)//2]  # pick a mid-chain strike

convergence = []
for n in sim_sizes:
    St = simulate_terminal(S, r, sigma, T, n, seed=42)
    price = mc_price(St, strike_check, r, T, "call")
    convergence.append({"n_sims": n, "MC Price": price})

conv_df = pd.DataFrame(convergence)
fig = px.line(conv_df, x="n_sims", y="MC Price", markers=True, log_x=True,
              title=f"MC Price Convergence at Strike {strike_check}")
fig.add_hline(y=black_scholes(S, strike_check, T, r, sigma, "call"),
              line_dash="dash", annotation_text="B-S Price")
fig.show()

In [ ]:
comparison = bs_prices.merge(mc_prices, on="Strike Price").merge(
    opt_chain[["strike", "Pmkt"]].rename(columns={"strike": "Strike Price"}),
    on="Strike Price",
    how="left",
)
comparison

In [ ]:
fig = px.line(
    comparison.dropna(subset=["Pmkt"]),
    x="Strike Price",
    y=["B-S Call", "MC Call", "Pmkt"],
    markers=True,
    title=f"{symbol} Model Price vs Market Price — Expiration {optexp.strftime('%Y-%m-%d')}",
)
fig.update_layout(yaxis_title="Option Price ($)")
fig.show()

In [ ]:
comparison["B-S Error"] = comparison["B-S Call"] - comparison["Pmkt"]
comparison["MC Error"] = comparison["MC Call"] - comparison["Pmkt"]
comparison["B-S Error %"] = comparison["B-S Error"] / comparison["Pmkt"]

fig = px.bar(
    comparison.dropna(subset=["Pmkt"]),
    x="Strike Price",
    y=["B-S Error", "MC Error"],
    barmode="group",
    title=f"{symbol} Pricing Error vs Market — B-S and MC",
)
fig.show()

In [ ]:
greeks_call = build_greeks_table(S, K, T, r, sigma, option_type="call")
greeks_put = build_greeks_table(S, K, T, r, sigma, option_type="put")

print(f"Greeks for {symbol} call")
display(greeks_call)

print(f"Greeks for {symbol} put")
display(greeks_put)

In [ ]:
fig = px.line(greeks_call, x="Strike Price", y=["Delta", "Gamma", "Vega"],
              markers=True, title=f"{symbol} Call Greeks Across Strikes")
fig.show()

In [ ]:
bs_iv = build_iv_table(S, opt_chain, T, r, option_type="call")
bs_iv

In [ ]:
mc_iv_records = []
for _, row in mc_prices.iterrows():
    strike = row["Strike Price"]
    mc_call_price = row["MC Call"]
    iv = iv_solve(S, strike, T, r, mc_call_price, option_type="call")
    mc_iv_records.append({"Strike Price": strike, "MC Price": mc_call_price, "MC Implied Vol": iv})

mc_iv_df = pd.DataFrame(mc_iv_records)
mc_iv_df

In [ ]:
strike_test = opt_chain["strike"].iloc[0]
pmkt_test = opt_chain["Pmkt"].iloc[0]

iv_test = iv_solve(S, strike_test, T, r, pmkt_test)
recovered_price = black_scholes(S, strike_test, T, r, iv_test, option_type="call")

print(f"Strike: {strike_test:.2f}")
print(f"Implied Vol: {iv_test:.3f}")
print(f"Historical sigma: {sigma:.3f}")
print(f"Recovered Price: {recovered_price:.2f} vs Pmkt: {pmkt_test:.2f}")

In [ ]:
import plotly.express as px

fig = px.line(
    bs_iv.dropna(subset=["IV"]),
    x="Strike Price",
    y="IV",
    markers=True,
    title=f"{symbol} Implied Volatility Smile — Expiration {optexp.strftime('%Y-%m-%d')}",
)
fig.update_layout(yaxis_tickformat=".1%")
fig.show()

In [ ]:
combined_iv = bs_iv[["Strike Price", "IV"]].rename(columns={"IV": "B-S Implied Vol"}).merge(
    mc_iv_df[["Strike Price", "MC Implied Vol"]], on="Strike Price"
)

fig = px.line(
    combined_iv,
    x="Strike Price",
    y=["B-S Implied Vol", "MC Implied Vol"],
    markers=True,
    title=f"{symbol} Market vs Monte Carlo Implied Volatility",
)
fig.update_layout(yaxis_tickformat=".1%")
fig.show()